In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from scipy.stats import norm

from model import MLP, CompressedMLP, CompressedMLP2
from data import generate_structured_data, generate_random_data, get_fashion_mnist, get_cifar10
from train_utils import train
from plot import *
from utils import *


In [2]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

cuda:0


In [3]:
relu = lambda X: np.maximum(X, 0)
d_relu = lambda X: np.maximum(np.sign(X), 0)

elu = lambda X, alpha=1.0: ( alpha * (np.exp(X) - 1) ) * (X <= 0) + (X * (X > 0))
d_elu = lambda X, alpha=1.0: ( alpha * np.exp(X) ) * (X <= 0) + (X > 0)

gelu = lambda X: X * norm.cdf(X)
d_gelu = lambda X: norm.cdf(X) + X * norm.pdf(X)

sigmoid = lambda X: 1 / (1 + np.exp(-X))
d_sigmoid = lambda X: sigmoid(X) * (1 - sigmoid(X))

silu = lambda X: X * sigmoid(X)
d_silu = lambda X: sigmoid(X) + X * d_sigmoid(X)

leaky_relu = lambda X, slope=0.01: slope * np.minimum(X, 0) + np.maximum(X, 0)
d_leaky_relu = lambda X, slope=0.01: slope * (X <= 0) + (X > 0).astype(float)

identity = lambda X: leaky_relu(X, slope=1)
d_identity = lambda X: d_leaky_relu(X, slope=1)

abs_ = lambda X: np.abs(X)
d_abs_ = lambda X: -1 * (X <= 0) + (X > 0)

softplus = lambda X: np.log(1 + np.exp(X))
d_softplus = lambda X: sigmoid(X)

smooth_abs = lambda X, beta = 10: (2 / beta) * softplus(X) - X - ( (2 / beta) * np.log(2) )
d_smooth_abs = lambda X, beta = 10: (2 / beta) * d_softplus(X) - 1

custom_activations_map = {'ReLU': [relu, d_relu],
               'ELU': [elu, d_elu],
               'GELU': [gelu, d_gelu],
               'SiLU': [silu, d_silu],
               'Leaky-ReLU': [leaky_relu, d_leaky_relu],
               'Abs': [abs_, d_abs_],
               'SmoothAbs': [smooth_abs, d_smooth_abs]}

In [4]:
reduction = 'sum'
loss_map = {
    'ce_loss': nn.CrossEntropyLoss(reduction=reduction),
    'mse_loss': nn.MSELoss(reduction=reduction)
}

# Synthetic Data 

In [ ]:
# Data setup
d = 32 # Ambient dim
h_d = d
p = d # Intrinsic dim 
K = 4 # Number of classes
N_k = 500
N = N_k * K
nu = 1

r = np.minimum(K, p)

# Model setup
L = 4
act_fn_str = 'ELU'
act, d_act = custom_activations_map[act_fn_str]
residual = False
batchnorm = False
init_method = 'orthogonal'
init_scale = 1
lr = 1e-4
gamma = 3

whiten_ = True
normalize_ = False

# Training setup
epochs = 200
freq = 1
#loss_fn = 'ce_loss'
loss_fn = 'mse_loss'

data_type = 'Structured'


num_trials = 1

In [ ]:
orig_losses = np.zeros((num_trials, epochs))
comp1_losses = np.zeros((num_trials, epochs))
comp2_losses = np.zeros((num_trials, epochs))

comp1_out_errors = np.zeros((num_trials, epochs))
comp2_out_errors = np.zeros((num_trials, epochs))

# comp1_feature_errors = np.zeros((num_trials, epochs))
# comp2_feature_errors = np.zeros((num_trials, epochs))
for i in range(num_trials):
    print("======== TRIAL", i + 1, "========")
    
    # Generate train samples
    if data_type == 'Structured':
        dataset, dataloader = generate_structured_data(d, p, K, N_k, nu=nu, whiten_=whiten_, normalize_=normalize_)
    elif data_type == 'Random':
        dataset, dataloader = generate_random_data(d, p, K, N_k, nu=nu, whiten_=whiten_, normalize_=normalize_)

    X = dataset.samples.T
    Y = dataset.labels.T

    # Train orig model
    print("Training original model.")
    model = MLP(L, d, h_d, K, act_fn_str, residual=residual, batchnorm=batchnorm, init_method=init_method, init_scale=init_scale)
    model = model.to(device)
    
    criterion = loss_map[loss_fn]
    
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0, weight_decay=0)
    
    weights, orig_init_weights, grads, outs, train_losses, _, _ = train(model, criterion, optimizer, dataloader, dataloder, epochs, freq, device)
    orig_losses[i, :] = train_losses
    
    # Calculate residual at initialization
    E_0 = outs[0].T - Y
    
    # Get update subspaces
    U1_null, V1_null = small_update_subspace_2(orig_init_weights, E_0, X, K, init_scale)
    U1 = complete_basis(U1_null)
    V1 = complete_basis(V1_null)
        
    # Train compressed model: same learning rate
    print("\nTraining compressed model: same learning rate. \n")
    comp1_model = CompressedMLP2(orig_init_weights, U1, V1, K, act_fn_str, init_scale)
    comp1_model = comp1_model.to(device)
    
    comp1_optimizer = optim.SGD(comp1_model.parameters(), lr=lr, momentum=0, weight_decay=0)
    
    comp1_weights, _, comp1_grads, comp1_outs, comp1_train_losses, _, _ = train(comp1_model, criterion, comp1_optimizer, dataloader, dataloder, epochs, freq, device)
    comp1_losses[i, :] = comp1_train_losses
    
    
    # Train compressed model: discrepant learning rate
    comp2_model = CompressedMLP2(orig_init_weights, U1, V1, K, act_fn_str, init_scale)
    comp2_model = comp2_model.to(device)
    
    comp2_optimizer = optim.SGD([{'params': [param.weight for param in comp2_model.comp_params]},
                                {'params': [param.weight for param in comp2_model.subspace_params], 'lr': gamma*lr}], 
                                lr=lr, momentum=0, weight_decay=0)

    comp2_weights, _, comp2_grads, comp2_outs, comp2_train_losses = train(comp2_model, criterion, comp2_optimizer, dataloader, epochs, freq, device)
    comp2_losses[i, :] = comp2_train_losses
    
    
    # Last-layer activation errors between original / comp_v1 and original / comp2 models
    comp1_out_err = np.array( [np.linalg.norm(outs[i] - comp1_outs[i], ord='fro')**2 / N for i in range(epochs)] ) #np.linalg.norm(activations[i], ord='fro') for i in range(epochs)] )
    comp2_out_err = np.array( [np.linalg.norm(outs[i] - comp2_outs[i], ord='fro')**2 / N for i in range(epochs)] ) #np.linalg.norm(activations[i], ord='fro') for i in range(epochs)] )
    
    comp1_out_errors[i, :] = comp1_out_err
    comp2_out_errors[i, :] = comp2_out_err
    

    comp1_feature_err = np.array( [np.linalg.norm(features[i] - comp1_features[i], ord='fro')**2 / N for i in range(epochs)] ) #np.linalg.norm(activations[i], ord='fro') for i in range(epochs)] )
    comp2_feature_err = np.array( [np.linalg.norm(features[i] - comp2_features[i], ord='fro')**2 / N for i in range(epochs)] ) #np.linalg.norm(activations[i], ord='fro') for i in range(epochs)] )
    
    comp1_feature_errors[i, :] = comp1_feature_err
    comp2_feature_errors[i, :] = comp2_feature_err
    

In [ ]:
# Mean / stdev of train losses
mean_orig_losses = np.mean(orig_losses, axis=0)
std_orig_losses = np.std(orig_losses, axis=0)

mean_comp1_losses = np.mean(comp1_losses, axis=0)
std_comp1_losses = np.std(comp1_losses, axis=0)

mean_comp2_losses = np.mean(comp2_losses, axis=0)
std_comp2_losses = np.std(comp2_losses, axis=0)


# Mean / stdev of activation errors
mean_comp1_out_errors = np.mean(comp1_out_errors, axis=0)
std_comp1_out_errors = np.std(comp1_out_errors, axis=0)

mean_comp2_out_errors = np.mean(comp2_out_errors, axis=0)
std_comp2_out_errors = np.std(comp2_out_errors, axis=0)


# Mean / stdev of second-to-last layer hidden features errors
# mean_comp1_feature_errors = np.mean(comp1_feature_errors, axis=0)
# std_comp1_feature_errors = np.std(comp1_feature_errors, axis=0)

# mean_comp2_feature_errors = np.mean(comp2_feature_errors, axis=0)
# std_comp2_feature_errors = np.std(comp2_feature_errors, axis=0)

In [ ]:
import os

savefig = False
save_dir = loss_fn + '_' + act_fn_str + '_activation_' + init_method + '_init_scale_' + str(init_scale) + '_L_' + str(L) + '_d_' + str(d) + '_p_' + str(p) + '_K_' + str(K)
if whiten_:
    save_path = os.path.join('./figs/whitened_data', save_dir)
elif normalize_ and not whiten_:
    save_path = os.path.join('./figs/normalized_data', save_dir)
else:
    save_path = os.path.join('./figs/raw_data', save_dir)

if savefig and not os.path.exists(save_path):
    os.makedirs(save_path)

# Plot train losses
plt.figure()
plt.plot(np.arange(epochs) + 1, mean_orig_losses, label='Uncompressed')
plt.fill_between(np.arange(epochs) + 1, mean_orig_losses - std_orig_losses, mean_orig_losses + std_orig_losses, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp1_losses, label='Compressed (same lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp1_losses - std_comp1_losses, mean_comp1_losses + std_comp1_losses, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp2_losses, label='Compressed (discrepant lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp2_losses - std_comp2_losses, mean_comp2_losses + std_comp2_losses, alpha=0.5)

plt.title(act_fn_str + ' Network Train Losses', fontsize=15) # (L = ' + str(L) + ', d = ' + str(d) + ', K = ' + str(K) + ')', fontsize=15)
plt.xlabel('Epoch (t)', fontsize=12)
plt.ylabel('Train Loss', fontsize=12)

plt.legend()
if savefig:
    save_fname = 'comp_uncomp_losses.pdf'
    plt.savefig(os.path.join(save_path, save_fname))
plt.tight_layout()
plt.show()
plt.close()


# Plot activation errors
plt.figure()
plt.plot(np.arange(epochs) + 1, mean_comp1_out_errors, label='Compressed (same lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp1_out_errors - std_comp1_out_errors, mean_comp1_out_errors + std_comp1_out_errors, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp2_out_errors, label='Compressed (discrepant lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp2_out_errors - std_comp2_out_errors, mean_comp2_out_errors + std_comp2_out_errors, alpha=0.5)

plt.title(act_fn_str + ' Network End-to-End Trajectories', fontsize=15) # (L = ' + str(L) + ', d = ' + str(d) + ', K = ' + str(K) + ')', fontsize=15)
plt.xlabel('Epoch (t)', fontsize=12)
plt.ylabel(r'Model Output Distance', fontsize=12)

plt.legend()
if savefig:
    save_fname = 'comp_uncomp_traj.pdf' 
    plt.savefig(os.path.join(save_path, save_fname))
plt.tight_layout()
plt.show()
plt.close()

#
# plt.figure()
# plt.plot(np.arange(epochs) + 1, mean_comp1_feature_errors, label='Compressed (same lr)')
# plt.fill_between(np.arange(epochs) + 1, mean_comp1_feature_errors - std_comp1_feature_errors, mean_comp1_feature_errors + std_comp1_feature_errors, alpha=0.5)

# plt.plot(np.arange(epochs) + 1, mean_comp2_feature_errors, label='Compressed (discrepant lr)')
# plt.fill_between(np.arange(epochs) + 1, mean_comp2_feature_errors - std_comp2_feature_errors, mean_comp2_feature_errors + std_comp2_feature_errors, alpha=0.5)

# plt.title(act_fn_str + ' Network Last Hidden Layer Features', fontsize=15) # (L = ' + str(L) + ', d = ' + str(d) + ', K = ' + str(K) + ')', fontsize=15)
# plt.xlabel('Epoch (t)', fontsize=12)
# plt.ylabel(r'Distance', fontsize=12)

# plt.legend()
# if savefig:
#     save_fname = 'comp_uncomp_features.pdf' 
#     plt.savefig(os.path.join(save_path, save_fname))
# plt.tight_layout()
# plt.show()
# plt.close()

In [ ]:
# print(comp2_features[-1], "\n")
# print(features[-1])

print("\n\n", comp2_model.layers[-1].weight.detach().cpu().numpy())
print("\n", model.layers[-1].weight.detach().cpu().numpy())

# Fashion MNIST 

In [ ]:
def transform_fashion_mnist(X: np.ndarray, mean: float, std: float):
    X = X.astype(float) / 255.
    X = (X - mean) / std
    
    return X

In [5]:
# Data setup
d = 28**2 
K = 5
N_k = 6000
N = N_k * K
data_dir = '/scratch/qingqu_root/qingqu1/alecx/fashion_mnist/'

# Model setup
L = 4
h_d = d
act_fn_str = 'ELU'
act, d_act = custom_activations_map[act_fn_str]
residual = False
batchnorm = False
init_method = 'orthogonal'
init_scale = 1e-1
gamma = 1


whiten_ = False
normalize_ = False

# Training setup
lr = 5e-4
lr_tilde = 1.4 * lr
epochs = 5
freq = 1
loss_fn = 'mse_loss'


num_trials = 1

In [7]:
orig_train_losses = np.zeros((num_trials, epochs))
comp1_train_losses = np.zeros((num_trials, epochs))
comp2_train_losses = np.zeros((num_trials, epochs))

orig_train_accs = np.zeros((num_trials, epochs))
comp1_train_accs = np.zeros((num_trials, epochs))
comp2_train_accs = np.zeros((num_trials, epochs))

orig_val_losses = np.zeros((num_trials, epochs))
comp1_val_losses = np.zeros((num_trials, epochs))
comp2_val_losses = np.zeros((num_trials, epochs))

orig_val_accs = np.zeros((num_trials, epochs))
comp1_val_accs = np.zeros((num_trials, epochs))
comp2_val_accs = np.zeros((num_trials, epochs))

comp1_out_errors = np.zeros((num_trials, epochs))
comp2_out_errors = np.zeros((num_trials, epochs))

comp1_feature_errors = np.zeros((num_trials, epochs))
comp2_feature_errors = np.zeros((num_trials, epochs))
for i in range(num_trials):
    print("======== TRIAL", i + 1, "========")
    
    # Generate train samples
    train_set, train_loader = get_fashion_mnist(K, N_k, data_dir, batch_size=N, shuffle=False)
    val_set, val_loader = get_fashion_mnist(K, N_k, data_dir, batch_size=N, train=False, shuffle=False)

    # Train orig model
    print("Training original model.")
    model = MLP(L, d, h_d, K, act_fn_str, residual=residual, batchnorm=batchnorm, init_method=init_method, init_scale=init_scale)
    #model = model.to(device)

    criterion = loss_map[loss_fn]
    
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0, weight_decay=0)
    #optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0)
    
    weights, orig_init_weights, grads, outs, train_losses, train_accs, val_losses, val_accs = train(model, criterion, optimizer, train_loader, val_loader, epochs, freq, device)
    orig_train_losses[i, :] = train_losses
    orig_train_accs[i, :] = train_accs
    orig_val_losses[i, :] = val_losses
    orig_val_accs[i, :] = val_accs
    
    # Calculate residual at initialization
    for data, _ in train_loader:
        X_in = data.detach().cpu().numpy().T
        break 
    Y = train_set.targets.T
    
    E_0 = outs[0].T - Y
    
    print(np.array(outs).shape)
    
    # Get update subspaces
    U1_null, V1_null = small_update_subspace_2(orig_init_weights, E_0, X_in, K, init_scale)
    #U1_null, V1_null = small_update_subspace(orig_init_weights[0], grads[0][0], K, init_scale)
    U1 = complete_basis(U1_null)
    V1 = complete_basis(V1_null)
        
    # Train compressed model: same learning rate
    print("\nTraining compressed model: same learning rate.")
    comp1_model = CompressedMLP2(orig_init_weights, U1, V1, K, act_fn_str, init_method, init_scale)
    #comp1_model = comp1_model.to(device)
    
    criterion = loss_map[loss_fn]
    comp1_optimizer = optim.SGD(comp1_model.parameters(), lr=lr, momentum=0, weight_decay=0)
    
    comp1_weights, _, comp1_grads, comp1_outs, comp1_train_loss, comp1_train_acc, comp1_val_loss, comp1_val_acc = train(comp1_model, criterion, comp1_optimizer, train_loader, val_loader, epochs, freq, device)
    comp1_train_losses[i, :] = comp1_train_loss
    comp1_train_accs[i, :] = comp1_train_acc
    comp1_val_losses[i, :] = comp1_val_loss
    comp1_val_accs[i, :] = comp1_val_acc
    
    # Last-layer activation errors between original / comp1 models
    comp1_out_err = np.array( [np.linalg.norm(outs[i] - comp1_outs[i], ord='fro')**2 / N for i in range(epochs)] ) #np.linalg.norm(activations[i], ord='fro') for i in range(epochs)] )
    comp1_out_errors[i, :] = comp1_out_err
    
    
    
    # Train compressed model: discrepant learning rate
    print("\nTraining compressed model: different learning rate. \n")
    comp2_model = CompressedMLP2(orig_init_weights, U1, V1, K, act_fn_str, init_method, init_scale)
    comp2_model = comp2_model.to(device)
    
    criterion = loss_map[loss_fn]
    comp2_optimizer = optim.SGD(comp2_model.parameters(), lr=lr_tilde, weight_decay=0)
#     comp2_optimizer = optim.SGD([{'params': [param.weight for param in comp2_model.comp_params]},
#                                 {'params': [param.weight for param in comp2_model.subspace_params], 'lr': gamma*lr}], 
#                                 lr=lr, momentum=0, weight_decay=0)
#     comp2_optimizer = optim.AdamW([{'params': [param.weight for param in comp2_model.comp_params]},
#                                 {'params': [param.weight for param in comp2_model.subspace_params], 'lr': gamma*lr}], 
#                                 lr=lr, weight_decay=0)

    comp2_weights, _, comp2_grads, comp2_outs, comp2_train_loss, comp2_train_acc, comp2_val_loss, comp2_val_acc = train(comp2_model, criterion, comp2_optimizer, train_loader, val_loader, epochs, freq, device)
    comp2_train_losses[i, :] = comp2_train_loss
    comp2_train_accs[i, :] = comp2_train_acc
    comp2_val_losses[i, :] = comp2_val_loss
    comp2_val_accs[i, :] = comp2_val_acc
    
    # Last-layer activation errors between original / comp2 models
    comp2_out_err = np.array( [np.linalg.norm(outs[i] - comp2_outs[i], ord='fro')**2 / N for i in range(epochs)] ) #np.linalg.norm(activations[i], ord='fro') for i in range(epochs)] )
    comp2_out_errors[i, :] = comp2_out_err
    

#     comp1_feature_err = np.array( [np.linalg.norm(features[i] - comp1_features[i], ord='fro')**2 / N for i in range(epochs)] ) #np.linalg.norm(activations[i], ord='fro') for i in range(epochs)] )
#     comp2_feature_err = np.array( [np.linalg.norm(features[i] - comp2_features[i], ord='fro')**2 / N for i in range(epochs)] ) #np.linalg.norm(activations[i], ord='fro') for i in range(epochs)] )
    
#     comp1_feature_errors[i, :] = comp1_feature_err
#     comp2_feature_errors[i, :] = comp2_feature_err
    

======== TRIAL 1 ========
Using Fashion MNIST train set with 5 classes!

Using Fashion MNIST val set with 5 classes!

Training original model.
Initializing weights using orthogonal method with scale 0.1


RuntimeError: CUDA error: invalid device ordinal
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
p = d

saveloss = False
save_dir = loss_fn + '_' + act_fn_str + '_activation_' + init_method + '_init_scale_' + str(init_scale) + '_L_' + str(L) + '_d_' + str(d) + '_p_' + str(p) + '_K_' + str(K)
save_path = os.path.join('./figs/fashion_mnist', save_dir)
if saveloss:
    np.savez(os.path.join(save_path, 'orig.npz'), orig_train_losses=orig_train_losses, orig_train_accs=orig_train_accs, orig_val_losses=orig_val_losses, orig_val_accs=orig_val_accs)
    np.savez(os.path.join(save_path, 'comp_same_lr.npz'), comp1_train_losses=comp1_train_losses, comp1_train_accs=comp1_train_accs, comp1_val_losses=comp1_val_losses, comp1_val_accs=comp1_val_accs, comp1_out_errors=comp1_out_errors)
    np.savez(os.path.join(save_path, 'comp_diff_lr.npz'), comp2_train_losses=comp2_train_losses, comp2_train_accs=comp2_train_accs, comp2_val_losses=comp2_val_losses, comp2_val_accs=comp2_val_accs, comp2_out_errors=comp2_out_errors)


# Mean / stdev of train losses
mean_orig_train_losses = np.mean(orig_train_losses, axis=0)
std_orig_train_losses = np.std(orig_train_losses, axis=0)

mean_comp1_train_losses = np.mean(comp1_train_losses, axis=0)
std_comp1_train_losses = np.std(comp1_train_losses, axis=0)

mean_comp2_train_losses = np.mean(comp2_train_losses, axis=0)
std_comp2_train_losses = np.std(comp2_train_losses, axis=0)


# Mean / stdev of train acc
mean_orig_train_accs = np.mean(orig_train_accs, axis=0)
std_orig_train_accs = np.std(orig_train_accs, axis=0)

mean_comp1_train_accs = np.mean(comp1_train_accs, axis=0)
std_comp1_train_accs = np.std(comp1_train_accs, axis=0)

mean_comp2_train_accs = np.mean(comp2_train_accs, axis=0)
std_comp2_train_accs = np.std(comp2_train_accs, axis=0)


# Mean / stdev of val losses
mean_orig_val_losses = np.mean(orig_val_losses, axis=0)
std_orig_val_losses = np.std(orig_val_losses, axis=0)

mean_comp1_val_losses = np.mean(comp1_val_losses, axis=0)
std_comp1_val_losses = np.std(comp1_val_losses, axis=0)

mean_comp2_val_losses = np.mean(comp2_val_losses, axis=0)
std_comp2_val_losses = np.std(comp2_val_losses, axis=0)


# Mean / stdev of val acc
mean_orig_val_accs = np.mean(orig_val_accs, axis=0)
std_orig_val_accs = np.std(orig_val_accs, axis=0)

mean_comp1_val_accs = np.mean(comp1_val_accs, axis=0)
std_comp1_val_accs = np.std(comp1_val_accs, axis=0)

mean_comp2_val_accs = np.mean(comp2_val_accs, axis=0)
std_comp2_val_accs = np.std(comp2_val_accs, axis=0)


# Mean / stdev of activation errors
mean_comp1_out_errors = np.mean(comp1_out_errors, axis=0)
std_comp1_out_errors = np.std(comp1_out_errors, axis=0)

mean_comp2_out_errors = np.mean(comp2_out_errors, axis=0)
std_comp2_out_errors = np.std(comp2_out_errors, axis=0)


# Mean / stdev of second-to-last layer hidden features errors
# mean_comp1_feature_errors = np.mean(comp1_feature_errors, axis=0)
# std_comp1_feature_errors = np.std(comp1_feature_errors, axis=0)

# mean_comp2_feature_errors = np.mean(comp2_feature_errors, axis=0)
# std_comp2_feature_errors = np.std(comp2_feature_errors, axis=0)

In [ ]:
import os

p = d

savefig = False
save_dir = loss_fn + '_' + act_fn_str + '_activation_' + init_method + '_init_scale_' + str(init_scale) + '_L_' + str(L) + '_d_' + str(d) + '_p_' + str(p) + '_K_' + str(K)
save_path = os.path.join('./figs/fashion_mnist', save_dir)

# Plot train losses
plt.figure()
plt.plot(np.arange(epochs) + 1, mean_orig_train_losses, label='Uncompressed train')
plt.fill_between(np.arange(epochs) + 1, mean_orig_train_losses - std_orig_train_losses, mean_orig_train_losses + std_orig_train_losses, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp1_train_losses, label='Compressed train (same lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp1_train_losses - std_comp1_train_losses, mean_comp1_train_losses + std_comp1_train_losses, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp2_train_losses, label='Compressed train (different lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp2_train_losses - std_comp2_train_losses, mean_comp2_train_losses + std_comp2_train_losses, alpha=0.5)


plt.plot(np.arange(epochs) + 1, mean_orig_val_losses, label='Uncompressed val')
plt.fill_between(np.arange(epochs) + 1, mean_orig_val_losses - std_orig_val_losses, mean_orig_val_losses + std_orig_val_losses, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp1_val_losses, label='Compressed val (same lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp1_val_losses - std_comp1_val_losses, mean_comp1_val_losses + std_comp1_val_losses, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp2_val_losses, label='Compressed val (different lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp2_val_losses - std_comp2_val_losses, mean_comp2_val_losses + std_comp2_val_losses, alpha=0.5)


plt.title(act_fn_str + ' Network Losses', fontsize=15) # (L = ' + str(L) + ', d = ' + str(d) + ', K = ' + str(K) + ')', fontsize=15)
plt.xlabel('Train Epoch (t)', fontsize=12)
plt.ylabel('Loss', fontsize=12)

plt.legend()
if savefig:
    save_fname = 'comp_uncomp_losses.pdf'
    plt.savefig(os.path.join(save_path, save_fname))
plt.tight_layout()
plt.show()
plt.close()


# Plot accuracies
plt.figure()
plt.plot(np.arange(epochs) + 1, mean_orig_train_accs, label='Uncompressed train')
plt.fill_between(np.arange(epochs) + 1, mean_orig_train_accs - std_orig_train_accs, mean_orig_train_accs + std_orig_train_accs, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp1_train_accs, label='Compressed (same lr) train')
plt.fill_between(np.arange(epochs) + 1, mean_comp1_train_accs - std_comp1_train_accs, mean_comp1_train_accs + std_comp1_train_accs, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp2_train_accs, label='Compressed (different lr) train')
plt.fill_between(np.arange(epochs) + 1, mean_comp2_train_accs - std_comp2_train_accs, mean_comp2_train_accs + std_comp2_train_accs, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_orig_val_accs, label='Uncompressed val')
plt.fill_between(np.arange(epochs) + 1, mean_orig_val_accs - std_orig_val_accs, mean_orig_val_accs + std_orig_val_accs, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp1_val_accs, label='Compressed (same lr) val')
plt.fill_between(np.arange(epochs) + 1, mean_comp1_val_accs - std_comp1_val_accs, mean_comp1_val_accs + std_comp1_val_accs, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp2_val_accs, label='Compressed (different lr) val')
plt.fill_between(np.arange(epochs) + 1, mean_comp2_val_accs - std_comp2_val_accs, mean_comp2_val_accs + std_comp2_val_accs, alpha=0.5)


plt.title(act_fn_str + ' Network Accuracies', fontsize=15) # (L = ' + str(L) + ', d = ' + str(d) + ', K = ' + str(K) + ')', fontsize=15)
plt.xlabel('Train Epoch (t)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)

plt.legend()
if savefig:
    save_fname = 'comp_uncomp_accs.pdf'
    plt.savefig(os.path.join(save_path, save_fname))
plt.tight_layout()
plt.show()
plt.close()



# Plot activation errors
plt.figure()
plt.plot(np.arange(epochs) + 1, mean_comp1_out_errors, label='Compressed (same lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp1_out_errors - std_comp1_out_errors, mean_comp1_out_errors + std_comp1_out_errors, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp2_out_errors, label='Compressed (different lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp2_out_errors - std_comp2_out_errors, mean_comp2_out_errors + std_comp2_out_errors, alpha=0.5)

plt.title(act_fn_str + ' Network End-to-End Trajectories', fontsize=15) # (L = ' + str(L) + ', d = ' + str(d) + ', K = ' + str(K) + ')', fontsize=15)
plt.xlabel('Train Epoch (t)', fontsize=12)
plt.ylabel(r'Model Output Distance', fontsize=12)

plt.legend()
if savefig:
    save_fname = 'comp_uncomp_traj.pdf' 
    plt.savefig(os.path.join(save_path, save_fname))
plt.tight_layout()
plt.show()
plt.close()

#
# plt.figure()
# plt.plot(np.arange(epochs) + 1, mean_comp1_feature_errors, label='Compressed (same lr)')
# plt.fill_between(np.arange(epochs) + 1, mean_comp1_feature_errors - std_comp1_feature_errors, mean_comp1_feature_errors + std_comp1_feature_errors, alpha=0.5)

# plt.plot(np.arange(epochs) + 1, mean_comp2_feature_errors, label='Compressed (discrepant lr)')
# plt.fill_between(np.arange(epochs) + 1, mean_comp2_feature_errors - std_comp2_feature_errors, mean_comp2_feature_errors + std_comp2_feature_errors, alpha=0.5)

# plt.title(act_fn_str + ' Network Last Hidden Layer Features', fontsize=15) # (L = ' + str(L) + ', d = ' + str(d) + ', K = ' + str(K) + ')', fontsize=15)
# plt.xlabel('Epoch (t)', fontsize=12)
# plt.ylabel(r'Distance', fontsize=12)

# plt.legend()
# if savefig:
#     save_fname = 'comp_uncomp_features.pdf' 
#     plt.savefig(os.path.join(save_path, save_fname))
# plt.tight_layout()
# plt.show()
# plt.close()

# CIFAR-10

In [ ]:
# Data setup
d = 3 * (32**2)
K = 4
N_k = 64
N = N_k * K
N_k_val = 1000
N_val = N_k_val * K
data_dir = '/scratch/qingqu_root/qingqu1/alecx/cifar10/'

# Model setup
L = 4
h_d = d
act_fn_str = 'ELU'
act, d_act = custom_activations_map[act_fn_str]
residual = False
batchnorm = False
init_method = 'orthogonal'
init_scale = 1e-1
gamma = 10

whiten_ = False
normalize_ = False

# Training setup
epochs = 500
loss_fn = 'mse_loss'
lr = 1e-4


num_trials = 1

In [ ]:
orig_losses = np.zeros((num_trials, epochs))
comp1_losses = np.zeros((num_trials, epochs))
comp2_losses = np.zeros((num_trials, epochs))

comp1_out_errors = np.zeros((num_trials, epochs))
comp2_out_errors = np.zeros((num_trials, epochs))

comp1_feature_errors = np.zeros((num_trials, epochs))
comp2_feature_errors = np.zeros((num_trials, epochs))
for i in range(num_trials):
    print("======== TRIAL", i + 1, "========")
    
    # Generate train samples
    train_set, train_loader = get_cifar10(K, N_k, data_dir, batch_size=N, shuffle=False)
    val_set, val_loader = get_cifar10(K, N_k_val, data_dir, batch_size=N_k_val, shuffle=False, train=False)

    # Train orig model
    print("Training original model.")
    model = MLP(L, d, h_d, K, act_fn_str, residual=residual, batchnorm=batchnorm, init_method=init_method, init_scale=init_scale)
    model = model.to(device)

    criterion = loss_map[loss_fn]
    
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0, weight_decay=0)
    
    weights, orig_init_weights, grads, outs, features, train_losses = train(model, criterion, optimizer, train_loader, val_loader, epochs, device)
    orig_losses[i, :] = train_losses
    
    # Calculate residual at initialization
    for data, _ in train_loader:
        X_in = data.detach().cpu().numpy().T
        break 
        
    E_0 = outs[0].T - Y
    
    # Get update subspaces
    U1_null, V1_null = small_update_subspace_2(orig_init_weights, E_0, X_in, K, init_scale)
    U1 = complete_basis(U1_null)
    V1 = complete_basis(V1_null)
        
    # Train compressed model: same learning rate
    print("\nTraining compressed model: same learning rate. \n")
    comp1_model = CompressedMLP2(orig_init_weights, U1, V1, K, act_fn_str, init_method, init_scale)
    comp1_model = comp1_model.to(device)
    
    comp1_optimizer = optim.SGD(comp1_model.parameters(), lr=lr, momentum=0, weight_decay=0)
    
    comp1_weights, _, comp1_grads, comp1_outs, comp1_features, comp1_train_losses = train(comp1_model, criterion, comp1_optimizer, train_loader, val_loader, epochs, device)
    comp1_losses[i, :] = comp1_train_losses
    
    
    # Train compressed model: discrepant learning rate
    comp2_model = CompressedMLP2(orig_init_weights, U1, V1, K, act_fn_str, init_method, init_scale)
    comp2_model = comp2_model.to(device)
    
    comp2_optimizer = optim.SGD([{'params': [param.weight for param in comp2_model.comp_params]},
                                {'params': [param.weight for param in comp2_model.subspace_params], 'lr': gamma*lr}], 
                                lr=lr, momentum=0, weight_decay=0)

    comp2_weights, _, comp2_grads, comp2_outs, comp2_features, comp2_train_losses = train(comp2_model, criterion, comp2_optimizer, train_loader, val_loader, epochs, device)
    comp2_losses[i, :] = comp2_train_losses
    
    
    # Last-layer activation errors between original / comp_v1 and original / comp2 models
    comp1_out_err = np.array( [np.linalg.norm(outs[i] - comp1_outs[i], ord='fro')**2 / N for i in range(epochs)] ) #np.linalg.norm(activations[i], ord='fro') for i in range(epochs)] )
    comp2_out_err = np.array( [np.linalg.norm(outs[i] - comp2_outs[i], ord='fro')**2 / N for i in range(epochs)] ) #np.linalg.norm(activations[i], ord='fro') for i in range(epochs)] )
    
    comp1_out_errors[i, :] = comp1_out_err
    comp2_out_errors[i, :] = comp2_out_err
    

    comp1_feature_err = np.array( [np.linalg.norm(features[i] - comp1_features[i], ord='fro')**2 / N for i in range(epochs)] ) #np.linalg.norm(activations[i], ord='fro') for i in range(epochs)] )
    comp2_feature_err = np.array( [np.linalg.norm(features[i] - comp2_features[i], ord='fro')**2 / N for i in range(epochs)] ) #np.linalg.norm(activations[i], ord='fro') for i in range(epochs)] )
    
    comp1_feature_errors[i, :] = comp1_feature_err
    comp2_feature_errors[i, :] = comp2_feature_err
    

In [ ]:
# Mean / stdev of train losses
mean_orig_losses = np.mean(orig_losses, axis=0)
std_orig_losses = np.std(orig_losses, axis=0)

mean_comp1_losses = np.mean(comp1_losses, axis=0)
std_comp1_losses = np.std(comp1_losses, axis=0)

mean_comp2_losses = np.mean(comp2_losses, axis=0)
std_comp2_losses = np.std(comp2_losses, axis=0)


# Mean / stdev of activation errors
mean_comp1_out_errors = np.mean(comp1_out_errors, axis=0)
std_comp1_out_errors = np.std(comp1_out_errors, axis=0)

mean_comp2_out_errors = np.mean(comp2_out_errors, axis=0)
std_comp2_out_errors = np.std(comp2_out_errors, axis=0)


# Mean / stdev of second-to-last layer hidden features errors
mean_comp1_feature_errors = np.mean(comp1_feature_errors, axis=0)
std_comp1_feature_errors = np.std(comp1_feature_errors, axis=0)

mean_comp2_feature_errors = np.mean(comp2_feature_errors, axis=0)
std_comp2_feature_errors = np.std(comp2_feature_errors, axis=0)

In [ ]:
import os

p = d

savefig = False
save_dir = loss_fn + '_' + act_fn_str + '_activation_' + init_method + '_init_scale_' + str(init_scale) + '_L_' + str(L) + '_d_' + str(d) + '_p_' + str(p) + '_K_' + str(K)
save_path = os.path.join('./figs/cifar10', save_dir)

# Plot train losses
plt.figure()
plt.plot(np.arange(epochs) + 1, mean_orig_losses, label='Uncompressed')
plt.fill_between(np.arange(epochs) + 1, mean_orig_losses - std_orig_losses, mean_orig_losses + std_orig_losses, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp1_losses, label='Compressed (same lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp1_losses - std_comp1_losses, mean_comp1_losses + std_comp1_losses, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp2_losses, label='Compressed (discrepant lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp2_losses - std_comp2_losses, mean_comp2_losses + std_comp2_losses, alpha=0.5)

plt.title(act_fn_str + ' Network Train Losses', fontsize=15) # (L = ' + str(L) + ', d = ' + str(d) + ', K = ' + str(K) + ')', fontsize=15)
plt.xlabel('Epoch (t)', fontsize=12)
plt.ylabel('Train Loss', fontsize=12)

plt.legend()
if savefig:
    save_fname = 'comp_uncomp_losses.pdf'
    plt.savefig(os.path.join(save_path, save_fname))
plt.tight_layout()
plt.show()
plt.close()


# Plot activation errors
plt.figure()
plt.plot(np.arange(epochs) + 1, mean_comp1_out_errors, label='Compressed (same lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp1_out_errors - std_comp1_out_errors, mean_comp1_out_errors + std_comp1_out_errors, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp2_out_errors, label='Compressed (discrepant lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp2_out_errors - std_comp2_out_errors, mean_comp2_out_errors + std_comp2_out_errors, alpha=0.5)

plt.title(act_fn_str + ' Network End-to-End Trajectories', fontsize=15) # (L = ' + str(L) + ', d = ' + str(d) + ', K = ' + str(K) + ')', fontsize=15)
plt.xlabel('Epoch (t)', fontsize=12)
plt.ylabel(r'Model Output Distance', fontsize=12)

plt.legend()
if savefig:
    save_fname = 'comp_uncomp_traj.pdf' 
    plt.savefig(os.path.join(save_path, save_fname))
plt.tight_layout()
plt.show()
plt.close()

#
plt.figure()
plt.plot(np.arange(epochs) + 1, mean_comp1_feature_errors, label='Compressed (same lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp1_feature_errors - std_comp1_feature_errors, mean_comp1_feature_errors + std_comp1_feature_errors, alpha=0.5)

plt.plot(np.arange(epochs) + 1, mean_comp2_feature_errors, label='Compressed (discrepant lr)')
plt.fill_between(np.arange(epochs) + 1, mean_comp2_feature_errors - std_comp2_feature_errors, mean_comp2_feature_errors + std_comp2_feature_errors, alpha=0.5)

plt.title(act_fn_str + ' Network Last Hidden Layer Features', fontsize=15) # (L = ' + str(L) + ', d = ' + str(d) + ', K = ' + str(K) + ')', fontsize=15)
plt.xlabel('Epoch (t)', fontsize=12)
plt.ylabel(r'Distance', fontsize=12)

plt.legend()
if savefig:
    save_fname = 'comp_uncomp_features.pdf' 
    plt.savefig(os.path.join(save_path, save_fname))
plt.tight_layout()
plt.show()
plt.close()